# <a id='toc1_'></a>[Coleta de Logs](#toc0_)

**Sumário**<a id='toc0_'></a>    
- [Coleta de Logs](#toc1_)    
  - [Introdução](#toc1_1_)    
  - [Desenvolvimento](#toc1_2_)    
    - [Coleta de logs por palavra-chave](#toc1_2_1_)    
    - [Tratando os dados com PLN](#toc1_2_2_)    
  - [Conclusão](#toc1_3_)    
  - [Referências](#toc1_4_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[Introdução](#toc0_)

Esse notebook tem como objetivo realizar a coleta de logs do sistema OLOG do LNLS - CNPEM. Para isso, utilizaremos o endpoint da API seguindo a referência [[1](#referencias)].

Nesse sistema, existem logs de diferentes registros dos operadaros da sala de controle do Sirius. No nosso caso, estamos interessados naqueles que se referenciam à abolos sísmicos. Para isso, iremos buscar por duas palavras-chaves principais: `"Terremoto"` e `"Sísmico"`.

Uma vez com a descrição dos logs em mãos, iremos utilizar um modelo de linguagem para interpretar a descrição e retornar os horários aproximados do início e do final do evento reportado. Faremos isso para não procurar cada um manualmente.

Por fim, guardaremos todas essas informações em uma planilha que será utilizada posteriormente para buscar os dados das PVs (Variáveis de Processo) do Sirius.

## <a id='toc1_2_'></a>[Desenvolvimento](#toc0_)

### <a id='toc1_2_1_'></a>[Coleta de logs por palavra-chave](#toc0_)

Para coletar os logs, vamos realizar uma requisição HTTP do tipo GET, através da biblioteca `requests` para o endpoint `https://ais-eng-srv-ca.cnpem.br/Olog/resources/logs`, passando os parâmetros *search* (termo de busca), *sort* (atributo de ordenação), *page* (número da página) e *limit* (limite de entradas por página) diretamente na url.

Vamos começar buscando as entradas com a palavra "terremoto".

In [6]:
import requests

url = "https://ais-eng-srv-ca.cnpem.br/Olog/resources/logs"

paramaters = {
    "search": "*terremoto*",
    "sort": "created",
    "page": 1,
    "limit": 400,
}

response = requests.get(url, params=paramaters)
response.status_code
response.url

'https://ais-eng-srv-ca.cnpem.br/Olog/resources/logs?search=%2Aterremoto%2A&sort=created&page=1&limit=400'

O código 200 indica que a requisição foi bem sucedida.

Agora podemos podemos guardar a informação obtida em um dataframe utilizando o método `json_normalize` do pandas, e filtrar apenas as colunas que vamos utilizar.

In [11]:
import pandas as pd

df = pd.json_normalize(response.json())
df = df.reindex(["id", "version", "owner", "createdDate", "eventStart", "description", "attachments"],axis=1)

df.head()

,id,version,owner,createdDate,eventStart,description,attachments
0,34548,2,lucas.ransato,1792696591000,1792679580000,#### Terremoto na China - Nagqu\n\n\n* **11h33...,[{'fileName': 'Screenshot from 2026-10-22 16-0...
1,34292,2,alex.cavalcante,1791606683000,1791579600000,**Oscilaçôes no feixe devido a abalos sísmicos...,"[{'fileName': 'QUAKE DETAILS.png', 'contentTyp..."
2,34291,1,vanusa.souza,1791580202000,1791580202000,### **Turno de usuários**\n\n\n\n\n- 04h23: Oc...,[]
3,34284,4,wagner.bento,1791575366000,1791570239000,#### Oscilação no feixe devido a ocorrencia de...,[{'fileName': 'Screenshot from 2026-10-09 16-2...
4,33387,2,vanusa.souza,1787257212000,1787257212000,##### **Oscilaçôes no feixe- possível influênc...,[{'fileName': 'Screenshot from 2026-08-20 16-1...


Vamos fazer o mesmo processo com o termo "sísmico" para capturar possíveis logs que escaparam da primeira busca. Então, basta concatenar o novo valor com com o DataFrame original, e salvar em um arquivo .csv que podemos utilizar depois.

In [ ]:
paramaters = {
    "search": "*sísmico*",
    "sort": "created",
    "page": 1,
    "limit": 400,
}

response = requests.get(url, params=paramaters)

df_2 = pd.json_normalize(response.json())

df_2 = df_2.reindex(["id", "version", "owner", "createdDate", "eventStart", "description", "attachments"],axis=1)

df = pd.concat([df, df_2], axis=0, ignore_index=True).drop_duplicates("id")

df.to_csv("olog_data.csv", index=False)

df

### <a id='toc1_2_2_'></a>[Tratando os dados com PLN](#toc0_)

Agora que temos uma lista com todos os logs com as palavras "terremoto" e "sísmico", precisamos de uma forma para conseguir o horário de início e de fim do evento.

A maioria dos registros apresenta essa informação na descrição. Veja o registro `34292`, por exemplo. Sua descrição é a seguinte:

>   **Oscilaçôes no feixe devido a abalos sísmicos no Panamá**
>   
>   * **17:25** Os BPMs horizontais apresentaram oscilações peculiares de abalos sísmico (terremotos). Informações de sites especializados mostraram que ocorreram várias réplicas no Panamá a partir das 17h25 (horário de Brasília). Tais réplicas continuam a ocorrer comos vê no print da tela do site USGS, porém os BPMs detectaram apenas as réplicas entre 17h25 e 18h00. Os detectores de vibração (sismógrafos do Sirius) apresentam dados inconsistentes.

Podemos observar que dentro da descrição conseguimos encontrar o horário de início e fim do evento registrado. A data pode ser obtida através do campo *eventStart*.

Para não precisarmos ler os logs 1 por 1, e definir essas datas, podemos automatizar o processo utilizando um modelo de linguagem natural.

Nesse caso, vamos utilizar a biblioteca da OpenAI, que suporta uma gama alta de modelos. Nós utilizamos o `gemini-3.5-flash-lite` da Google, mas outros modelos também poderiam servir.

*Nota:* Perceba que a chave de API deve ser inserida ao rodar a célula abaixo

In [35]:
import getpass
from openai import OpenAI
import time
from datetime import datetime, timezone
import json

BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
API_KEY = getpass.getpass("Insira a chave API: ")
MODEL_NAME = "models/gemini-3.5-flash-lite"

Agora vamos criar um cliente para utilizar nosso modelo

In [14]:
client = OpenAI(
    base_url=BASE_URL, 
    api_key=API_KEY
)

Podemos testar o modelo enviando um simples *"Olá"* com o método `client.chat.completions.create()`.

In [15]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Olá"
        }
    ]
)

print(response.choices[0].message.content)

Olá! Como posso ajudar você hoje?


Podemos ver que o modelo consegue receber e processar a mensagem tranquilamente.

Agora vamos definir um `SYSTEM_PROMPT` para determinar o modo que o nosso modelo deve agir e identificar os horários dado um determinado log.

In [16]:
SYSTEM_PROMPT = """
You are a timestamp extraction engine.

TASK
Extract the event start and end times from the provided log.

INPUT

You will receive:
- Reference date (YYYY-MM-DD)
- Log text

RULES

1. Use ONLY information present in the log.
2. The reference date must always be used when a time appears without a date.
3. Assume timestamps are GMT-3 unless another timezone is explicitly specified.
4. Convert all timestamps to UTC.
5. Return timestamps in ISO 8601 format:
   YYYY-MM-DDTHH:MM:SS.000000Z

6. If an explicit time interval is found:
   - first time -> from
   - second time -> to

7. If only a single timestamp is found:
   - set "from" to that timestamp
   - set "to" to exactly 2 hours after "from"

8. If multiple timestamps are mentioned but they do not explicitly define an interval:
   - use the earliest timestamp as "from"
   - use the latest timestamp as "to"

9. If no timestamp can be determined:
   - from = null
   - to = null

10. Never return explanations.
11. Never return examples.
12. Never repeat the input.
13. Never return markdown.
14. Return only a valid JSON object.
15. All timestamps must always contain a complete ISO 8601 UTC datetime.

OUTPUT FORMAT

{
  "from": string | null,
  "to": string | null
}
"""

Por fim, basta criar uma função que recebe um log e processa a mensagem

In [33]:
def get_times_from_log(log):
    ts = float(log["eventStart"]) / 1000
    
    iso = datetime.fromtimestamp(
        ts,
        tz=timezone.utc
    ).strftime("%Y-%m-%dT%H:%M:%S.000000Z")
        
    message = f"Reference Date: {iso} | Description: {log["description"]}"
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": message
            }
        ],
        response_format={"type": "json_object"}
    )
    
    return response.choices[0].message.content

log_teste = df.loc[df["id"] == 34292].iloc[0]

print(log_teste["description"])
print()
print(get_times_from_log(log_teste))

**Oscilaçôes no feixe devido a abalos sísmicos no Panamá**

* **17:25**Os BPMs horizontais apresentaram oscilações peculiares de abalos sísmico (terremotos). Informações de sites especializados mostraram que ocorreram várias réplicas no Panamá a partir das 17h25 (horário de Brasília). Tais réplicas continuam a ocorrer comos vê no print da tela do site USGS, porém os BPMs detectaram apenas as réplicas entre 17h25 e 18h00. Os detectores de vibração (sismógrafos do Sirius) apresentam dados inconsistentes.

{
  "from": "2026-10-09T20:25:00.000000Z",
  "to": "2026-10-09T21:00:00.000000Z"
}


Como podermos ver, o modelo determina corretamente a janela entre as `17h25` e `18h00` do dia 9 de outubro de 2026. (Lembrando que o horário devolvido pelo modelo está no UTC 0, ou seja, devemos subtrair 3 horas para lê-lo como o horário de Brasília GMT-3).

Vamos criar uma função que escreve os tempos para todos os logs então.

*Nota:* Utilizamos um delay no tempo para não consumir o limite de requisições por minuto da API. Caso seu modelo não possua essa limitação, pode desconsiderá-la sem problemas.

In [34]:
times_from_logs = []

for i, log in df.iterrows():
    print(f"LOG {i:2} - {log["id"]}: Processando descrição...")
    
    start_time = time.time()
    response = get_times_from_log(log)
    delta_time = time.time() - start_time
    
    print(f"LOG {i:2} - {log["id"]}: Processado em {delta_time:.2f}s")
    time.sleep(6)
    
    times_from_logs.append(response)

LOG  0 - 34548: Processando descrição...
LOG  0 - 34548: Processado em 1.44s
LOG  1 - 34292: Processando descrição...
LOG  1 - 34292: Processado em 1.54s
LOG  2 - 34291: Processando descrição...
LOG  2 - 34291: Processado em 0.94s
LOG  3 - 34284: Processando descrição...
LOG  3 - 34284: Processado em 0.88s
LOG  4 - 33387: Processando descrição...
LOG  4 - 33387: Processado em 0.86s
LOG  5 - 33338: Processando descrição...
LOG  5 - 33338: Processado em 0.85s
LOG  6 - 33266: Processando descrição...
LOG  6 - 33266: Processado em 0.85s
LOG  7 - 33005: Processando descrição...
LOG  7 - 33005: Processado em 0.87s
LOG  8 - 32807: Processando descrição...
LOG  8 - 32807: Processado em 0.82s


KeyboardInterrupt: 

Agora, basta armazenar os valores encontrados junto da planilha original criando duas novas colunas: `from` e `to`.

In [ ]:
json_times_from_logs = [json.loads(x) for x in times_from_logs]
json_times_from_logs

df["from"] = [x["from"] for x in json_times_from_logs]
df["to"] = [x["to"] for x in json_times_from_logs]
df.to_csv("olog_data.csv", index=False)

df

Para os que não foram encontrados, vamos definir manualmente os tempos

In [ ]:
new_df = df.copy()

new_df.loc[8, "from"] = "2024-06-26T22:46:38.000000Z"
new_df.loc[8, "to"]   = "2024-06-27T00:46:38.000000Z"

new_df = new_df.drop(35, )

new_df.loc[36, "from"] = "2025-10-31T18:51:12.000000Z"
new_df.loc[36, "to"]   = "2025-10-31T20:51:12.000000Z"

new_df.loc[38, "from"] = "2025-09-18T19:00:00.000000Z"
new_df.loc[38, "to"]   = "2025-09-18T23:00:00.000000Z"

new_df.loc[44, "from"] = "2024-12-05T19:00:00.000000Z"
new_df.loc[44, "to"]   = "2024-12-05T23:00:00.000000Z"

new_df.to_csv("olog_data.csv", index=False)

new_df

## <a id='toc1_3_'></a>[Conclusão](#toc0_)

Por fim, temos um arquivo csv que contém os registros de logs do sistema olog e o tempo de início e fim de cada evento. Esse será a base de dados que utilizaremos inicialmente para encontrar os terremotos que interferiram significativamente no Sirius.

<span id="referencias"></span>
## <a id='toc1_4_'></a>[Referências](#toc0_)

1. PHOEBUS OLOG. *Rest HTTP API*. [S. l.]: Phoebus Olog Documentation, [s. d.]. Disponível em: https://olog.readthedocs.io/en/latest/developer/references/http-api.html. Acesso em: 23 out. 2026.